# Bronze: Vendas

**Objetivo:** copiar fielmente a tabela Raw de Vendas para a camada Bronze, sem nenhuma transformação de conteúdo — apenas adicionando metadados de controle. Mantém a Bronze como fonte da verdade bruta e imutável para a fato de vendas.

**Fonte:** tabela Delta `raw.vendas`.

**Destino:** tabela Delta `bronze.vendas`, particionada por `data_ingestao_particao` (mesma estratégia de particionamento da Raw, preservando eficiência de leitura/expurgo por data).

**Modo de execução:** streaming em batch (`trigger(availableNow=True)`), lendo incrementalmente de `raw.vendas` via Structured Streaming sobre Delta.

**Metadados adicionados:** `data_ingestao_bronze` (timestamp de chegada na Bronze).

**Observação de arquitetura:** nenhum casting de tipo, cálculo de valor total ou conversão de câmbio é realizado aqui — os preços continuam como texto. Essas responsabilidades pertencem exclusivamente à Silver.

In [0]:
# Imports

from pyspark.sql.functions import current_timestamp

print("Imports realizados com sucesso.")

In [0]:
# Bronze vendas

caminho_checkpoint_bronze_vendas = "/Volumes/poc_latam_food/landing/blob_simulado/_checkpoints/bronze/vendas"

df_stream_bronze_vendas = (
    spark.readStream
    .format("delta")
    .table("poc_latam_food.raw.vendas")
    .withColumn("data_ingestao_bronze", current_timestamp())
)

(
    df_stream_bronze_vendas
    .writeStream
    .format("delta")
    .option("checkpointLocation", caminho_checkpoint_bronze_vendas)
    .partitionBy("data_ingestao_particao")
    .trigger(availableNow=True)
    .toTable("poc_latam_food.bronze.vendas")
)

print("Bronze de Vendas concluído.")

In [0]:
# Validação visual - bronze.vendas

print("=== bronze.vendas ===")
spark.table("poc_latam_food.bronze.vendas").display()

print(f"Total de linhas: {spark.table('poc_latam_food.bronze.vendas').count()}")

print("Contagem por país:")
spark.table("poc_latam_food.bronze.vendas").groupBy("pais").count().display()